# 01 — Explore and clean the data

This notebook loads the 2009–2010 sheet, measures quality issues, and **writes the rules** later notebooks will use for table Q&A.

No Gemini calls here. Run from this `notebooks/` folder or from the repo root; the first code cell finds the project root either way.

Excel load can take ~30–60 seconds on this file.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import (
    add_analysis_flags,
    identified_customers,
    load_raw,
    return_rows,
    sales_rows,
    save_processed,
)

pd.set_option("display.max_columns", 20)

## Load and schema

UCI Online Retail II, sheet `Year 2009-2010`: invoice lines for a UK gift-ware retailer.

In [2]:
raw = load_raw()
print("shape:", raw.shape)
print("date range:", raw["InvoiceDate"].min(), "→", raw["InvoiceDate"].max())
display(raw.head())
print(raw.dtypes)

shape: (525461, 8)
date range: 2009-12-01 07:45:00 → 2010-12-09 20:01:00


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object


## Missing values

`Description` and `Customer ID` are the only incomplete columns. Every missing description is also a guest checkout (no customer id).

In [3]:
missing = raw.isna().sum()
display(missing.to_frame("missing_rows"))
print(
    "missing description and missing Customer ID:",
    int((raw["Description"].isna() & raw["Customer ID"].isna()).sum()),
)

,missing_rows
Invoice,0
StockCode,0
Description,2928
Quantity,0
InvoiceDate,0
Price,0
Customer ID,107927
Country,0


missing description and missing Customer ID: 2928


## Cancellations, returns, and odd prices

- Invoices whose id starts with `C` are cancellations.
- Quantity can be negative **without** a `C` prefix (stock corrections).
- Three rows are bad-debt adjustments (`StockCode = B`, invoice starts with `A`) with large **negative prices**.
- `Price == 0` occurs on a few thousand rows (free items / data issues).

In [4]:
invoice = raw["Invoice"].astype(str)
print("cancellation rows:", int(invoice.str.startswith("C").sum()))
print("unique cancellation invoices:", invoice[invoice.str.startswith("C")].nunique())
print("negative quantity:", int((raw["Quantity"] < 0).sum()))
print(
    "negative quantity but not a C invoice:",
    int(((raw["Quantity"] < 0) & ~invoice.str.startswith("C")).sum()),
)
print("zero price:", int((raw["Price"] == 0).sum()))
print("negative price:")
display(
    raw.loc[
        raw["Price"] < 0,
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"],
    ]
)

cancellation rows: 10206
unique cancellation invoices: 4592
negative quantity: 12326
negative quantity but not a C invoice: 2121
zero price: 3687
negative price:


,Invoice,StockCode,Description,Quantity,Price,Customer ID
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN


## Guests vs identified customers

About one in five lines has no `Customer ID`. Those are **guest checkouts**. They still count in country/product revenue, but they must not be treated as a customer in CLV or “unique customers”.

In [5]:
print("guest rows:", int(raw["Customer ID"].isna().sum()))
print(
    "guest invoices:",
    raw.loc[raw["Customer ID"].isna(), "Invoice"].astype(str).nunique(),
)
print("distinct Customer ID values (including return-only ids):", raw["Customer ID"].nunique(dropna=True))
print("countries:", raw["Country"].nunique())
display(raw["Country"].value_counts().head(8).to_frame("rows"))

guest rows: 107927
guest invoices: 5229
distinct Customer ID values (including return-only ids): 4383
countries: 40


,rows
Country,
United Kingdom,485852
EIRE,9670
Germany,8129
France,5772
Netherlands,2769
Spain,1278
Switzerland,1187
Portugal,1101


## Revenue and flags

`Revenue = Quantity * Price` (returns contribute negative revenue). Flags are added in `src/cleaning.py` so notebook 02 does not redo this logic.

In [8]:
df = add_analysis_flags(raw)
sold = sales_rows(df)
returned = return_rows(df)
buyers = identified_customers(df)

print("flagged shape:", df.shape)
print("gross revenue (all rows):", round(df["Revenue"].sum(), 2))
print("sales-row revenue (qty>0, price>0, not test/bad-debt):", round(sold["Revenue"].sum(), 2))
print("return/cancel rows:", len(returned), "revenue:", round(returned["Revenue"].sum(), 2))
print("identified customers with at least one sale:", len(buyers))
display(df.head())

flagged shape: (525461, 14)
gross revenue (all rows): 9539484.63
sales-row revenue (qty>0, price>0, not test/bad-debt): 10306039.59
return/cancel rows: 12327 revenue: -629855.37
identified customers with at least one sale: 4312


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancellation,is_guest,is_bad_debt,is_test,is_non_product,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,False,False,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,False,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,False,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,False,False,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,False,False,False,False,30.0


## Rules for later notebooks (the contract)

These defaults exist so gold-set answers and the pandas engine use the **same grain**.

1. **Line revenue** is `Quantity * Price`. Invoice revenue is the sum of line revenue for that `Invoice`.
2. **Cancellations** are `Invoice` ids starting with `C`. Treat them as returns. Also treat `Quantity < 0` as a return even when the invoice is not prefixed with `C`.
3. **Guest checkouts** (`Customer ID` null) stay in revenue, country, and product totals. They are **excluded** from unique-customer counts and customer lifetime value.
4. **Unique customers** = distinct `Customer ID` on **sales rows** (quantity > 0, price > 0, not test, not bad-debt). On this sheet that is **4,312** (not every distinct id: 69 ids only appear on negative lines).
5. **Customer lifetime value** (this dataset, this period) = sum of line revenue per identified customer, then average across those customers. Decide per question whether returns are netted; **default is net** (use all flagged rows for that customer, including cancellations).
6. **Bad-debt adjustments** (`StockCode == "B"` / invoice `A…`) are accounting rows, not sales. Exclude them from “largest sale” and product rankings unless the question is about adjustments.
7. **Postage, manual, discounts, bank charges, Amazon fees, gifts, TEST** are flagged `is_non_product`. Exclude them from “which product…” questions; keep them for total revenue unless the question says otherwise.
8. Do **not** drop rows from the master table. Filter in the query (or with `sales_rows` / `return_rows`) so we can still answer refund questions.

Naive RAG later will embed line text and **will not** see these global aggregates. That is the point of notebook 03.

In [ ]:
path = save_processed(df)
print("wrote", path, "(", path.stat().st_size // 1024, "KB )")
print("reload check:", pd.read_pickle(path).shape)